## load and prepare data

In [ ]:
%cd ../..
%matplotlib inline

import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import seaborn as sns
import statsmodels.stats.contingency_tables as sm_contigency_tables
from scipy.stats import wilcoxon
from statsmodels.stats.proportion import proportion_confint
from statannotations.Annotator import Annotator
from statsmodels.stats.multitest import multipletests

import sys
sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import (GROUP_3T, GROUP_3T_FLAIR, GROUP_7T_ADAPTED, GROUP_7T_ADAPTED_FLAIR,
                            GROUP_7T_DEFAULT, GROUP_3T_UNION_7T, load_and_prepare_data,
                            filter_to_common_subjects, harmo_labels, main_group_hue_family,
                            harmo_saturation, _with_saturation, _with_value)


In [ ]:
eval_stats_df = load_and_prepare_data()

## sensitivity and specificity

In [ ]:
# for two given analysis groups (e.g. analysis_group='3T' and analysis_group='7T_adapted'), for each model, harmo, site_subj_id,
# create a new analysis group that is the logical OR of both
# for the columns recon_successful, tp_patient and tn_control
# 
# copy all other columns from the first condition, the following should be set to np.nan:
# input_path, prediction_path, ground_truth_path, pred_volume, gt_volume, intersection_volume,
# dice, num_gt_clusters, num_pred_clusters, tp_clusters, fp_clusters, max_cluster_dice
# and min_cluster_distance
#
# e.g.: for '3T' and '7T_adapted', the new analysis group will be '3T-union-7T_adapted'
#       if tp_patient is 1 for either condition, it will be 1 for the new condition

def create_logical_union_condition(df, condition1, condition2):
    cols_recomputed = ['analysis_group', 'recon_successful', 'tp_patient', 'tn_control', 'fp_clusters']

    merge_on = ['model', 'harmo', 'site_subj_id', 'source', 'site', 'category_FCD', 'category_3T_MR_negative', 'category_7T_MR_negative']

    columns_to_nan = [c for c in df.columns if c not in set(cols_recomputed + merge_on)]

    df_condition1 = df[df['analysis_group'] == condition1].copy()
    df_condition2 = df[df['analysis_group'] == condition2].copy()

    df_condition1[columns_to_nan] = np.nan
    df_condition2[columns_to_nan] = np.nan

    df_merged = pd.merge(
        df_condition1,
        df_condition2,
        on=merge_on + columns_to_nan,
        suffixes=(None, '_merged'),
        how='outer',
        validate='one_to_one'
    )

    if len(df_merged) != len(df_condition1) or len(df_merged) != len(df_condition2):
        raise ValueError(f"Merge of conditions {condition1} and {condition2} did not result in the same number of rows as the original conditions. "
                         f"len(df_condition1)={len(df_condition1)}, len(df_condition2)={len(df_condition2)}, len(df_merged)={len(df_merged)}")

    new_condition = f"{condition1}-union-{condition2}"

    def logical_or_with_nan(a, b):
        if pd.isna(a) and pd.isna(b):
            return np.nan
        if pd.isna(a):
            return b
        if pd.isna(b):
            return a
        return a or b

    def logical_and_with_nan(a, b):
        if pd.isna(a) and pd.isna(b):
            return np.nan
        if pd.isna(a):
            return b
        if pd.isna(b):
            return a
        return a and b

    # create new columns for the logical OR of recon_successful, tp_patient and tn_control
    df_merged['recon_successful'] = df_merged.apply(lambda row: logical_or_with_nan(row['recon_successful'], row['recon_successful_merged']), axis=1)
    df_merged['tp_patient'] = df_merged.apply(lambda row: logical_or_with_nan(row['tp_patient'], row['tp_patient_merged']), axis=1)
    df_merged['tn_control'] = df_merged.apply(lambda row: logical_and_with_nan(row['tn_control'], row['tn_control_merged']), axis=1)    
    df_merged['fp_clusters'] = df_merged['fp_clusters'] + df_merged['fp_clusters_merged']

    # overwrite the analysis_group column with the new condition
    df_merged['analysis_group'] = new_condition

    # drop all columns ending in _merged
    df_merged = df_merged[[col for col in df_merged.columns if not col.endswith('_merged')]]

    return df_merged

eval_stats_df = pd.concat([eval_stats_df, create_logical_union_condition(eval_stats_df, GROUP_3T, GROUP_7T_ADAPTED)])

In [ ]:
def metric_with_ci(df, outcome_col, alpha=0.05):
    # only patients should have tp_patient set and only controls should have tn_control set
    # otherwise this is np.nan and will be dropped
    df_filtered = df[~pd.isna(df[outcome_col])][['site_subj_id', outcome_col]].dropna(subset=[outcome_col])
    n = len(df_filtered)
    k = df_filtered[outcome_col].sum()
    p = k / n if n > 0 else np.nan
    ci_low, ci_high = proportion_confint(k, n, alpha=alpha, method='wilson') if n > 0 else (np.nan, np.nan)
    return df_filtered, n, k, p, ci_low, ci_high

def mean_with_ci(df, count_col, n_boot=10000, alpha=0.05, seed=0):
    values = df[count_col].dropna().to_numpy()
    n = len(values)
    mean = values.mean() if n > 0 else np.nan
    if n == 0:
        return n, mean, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_means = rng.choice(values, size=(n_boot, n), replace=True).mean(axis=1)
    ci_low, ci_high = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return n, mean, ci_low, ci_high

def mcnemar_paired(df1, df2, outcome_col):
    # match subjects present in both conditions to get paired binary outcomes
    merged = pd.merge(df1, df2,
                      on='site_subj_id', 
                      suffixes=('_1', '_2'),
                      how='inner',
                      validate='one_to_one')
    # if subjects were dropped due to the merge, print a warning
    if len(merged) < min(len(df1), len(df2)):
        print(f"Warning: {len(df1) - len(merged)} subjects in df1 and {len(df2) - len(merged)} subjects in df2 were dropped due to no match on site_subj_id.")

    outcome_col_1, outcome_col_2 = f'{outcome_col}_1', f'{outcome_col}_2'
    merged = merged.dropna(subset=[outcome_col_1, outcome_col_2])
    both = int(((merged[outcome_col_1] == 1) & (merged[outcome_col_2] == 1)).sum())
    only1 = int(((merged[outcome_col_1] == 1) & (merged[outcome_col_2] == 0)).sum())
    only2 = int(((merged[outcome_col_1] == 0) & (merged[outcome_col_2] == 1)).sum())
    neither = int(((merged[outcome_col_1] == 0) & (merged[outcome_col_2] == 0)).sum())
    table = [[both, only1], [only2, neither]]
    result = sm_contigency_tables.mcnemar(table, exact=True)
    return merged.shape[0], table, result.statistic, result.pvalue

In [ ]:
def build_performance_table(eval_stats_df, analysis_groups, harmo_conditions):
    # sensitivity/specificity (+ 95%-CI, and a formatted "_display" string of both) for every
    # (harmo, analysis_group) combination in the given lists
    eval_stats_df = filter_to_common_subjects(eval_stats_df, analysis_groups, harmo_conditions)

    # we need to compute these before filtering to only include recon_successful==1
    failed_recons = {}
    for harmo, analysis_group in itertools.product(harmo_conditions, analysis_groups):
        failed_recons[(harmo, analysis_group, 'nsubjects')] = ((eval_stats_df['harmo'] == harmo) & 
                                                     (eval_stats_df['analysis_group'] == analysis_group)).sum()
        failed_recons[(harmo, analysis_group, 'all')] = ((eval_stats_df['harmo'] == harmo) & 
                                              (eval_stats_df['analysis_group'] == analysis_group) & 
                                              (eval_stats_df['recon_successful'] == 0)).sum()
        failed_recons[(harmo, analysis_group, 'patients')] = ((eval_stats_df['harmo'] == harmo) &
                                                   (eval_stats_df['analysis_group'] == analysis_group) &
                                                   (eval_stats_df['group'] == 'patient') &
                                                   (eval_stats_df['recon_successful'] == 0)).sum()
        failed_recons[(harmo, analysis_group, 'controls')] = ((eval_stats_df['harmo'] == harmo) &
                                                   (eval_stats_df['analysis_group'] == analysis_group) &
                                                   (eval_stats_df['group'] == 'control') &
                                                   (eval_stats_df['recon_successful'] == 0)).sum()

    # also keep a copy that retains subjects with failed reconstructions, with a binary recon_failed
    # column, used below for both the raw counts and the Wilson CI on the failed-recon rate
    eval_stats_df_recon = eval_stats_df.copy()
    eval_stats_df_recon['recon_failed'] = (eval_stats_df_recon['recon_successful'] == 0).astype(int)

    # now filter to only include subjects that were successfully reconstructed (recon_successful==1)
    eval_stats_df = eval_stats_df[eval_stats_df['recon_successful'] == 1]
    eval_stats_df = filter_to_common_subjects(eval_stats_df, analysis_groups, harmo_conditions)

    def format_metric(value, ci_low, ci_high):
        if pd.isna(value):
            return 'n/a'
        return f"{value * 100:.1f}% (95% CI: {ci_low * 100:.1f}-{ci_high * 100:.1f})"

    rows = []
    for harmo, analysis_group in itertools.product(harmo_conditions, analysis_groups):
        df_condition = eval_stats_df[(eval_stats_df['harmo'] == harmo) & (eval_stats_df['analysis_group'] == analysis_group)]
        df_condition_mrn = df_condition[df_condition['category_3T_MR_negative'] == 1]
        df_condition_recon = eval_stats_df_recon[(eval_stats_df_recon['harmo'] == harmo) & (eval_stats_df_recon['analysis_group'] == analysis_group)]

        _, n_patients, k_patients, sensitivity, sens_ci_low, sens_ci_high = metric_with_ci(df_condition, 'tp_patient')
        _, n_mrnpatients, k_mrnpatients, sensitivity_mrn, sens_mrn_ci_low, sens_mrn_ci_high = metric_with_ci(df_condition_mrn, 'tp_patient')
        _, n_controls, k_controls, specificity, spec_ci_low, spec_ci_high = metric_with_ci(df_condition, 'tn_control')
        _, n_recon, k_recon, failed_recon_rate, recon_ci_low, recon_ci_high = metric_with_ci(df_condition_recon, 'recon_failed')
        n_fp_clusters, mean_no_fp_clusters, fp_ci_low, fp_ci_high = mean_with_ci(df_condition, 'fp_clusters')

        rows.append({
            'harmo': harmo,
            'analysis_group': analysis_group,
            'n_subjects': failed_recons[(harmo, analysis_group, 'nsubjects')],
            'n_subjects_failed_recon': failed_recons[(harmo, analysis_group, 'all')],
            'n_patients_failed_recon': failed_recons[(harmo, analysis_group, 'patients')],
            'n_controls_failed_recon': failed_recons[(harmo, analysis_group, 'controls')],
            'failed_recon_rate': failed_recon_rate,
            'failed_recon_rate_ci_low': recon_ci_low,
            'failed_recon_rate_ci_high': recon_ci_high,
            'failed_recon_rate_display': format_metric(failed_recon_rate, recon_ci_low, recon_ci_high),
            'n_patients': n_patients,
            'k_patients': k_patients,
            'sensitivity': sensitivity,
            'sensitivity_ci_low': sens_ci_low,
            'sensitivity_ci_high': sens_ci_high,
            'sensitivity_display': format_metric(sensitivity, sens_ci_low, sens_ci_high),
            'n_mrnpatients': n_mrnpatients,
            'k_mrnpatients': k_mrnpatients,
            'sensitivity_mrn': sensitivity_mrn,
            'sensitivity_mrn_ci_low': sens_mrn_ci_low,
            'sensitivity_mrn_ci_high': sens_mrn_ci_high,
            'sensitivity_mrn_display': format_metric(sensitivity_mrn, sens_mrn_ci_low, sens_mrn_ci_high),
            'n_controls': n_controls,
            'k_controls': k_controls,
            'specificity': specificity,
            'specificity_ci_low': spec_ci_low,
            'specificity_ci_high': spec_ci_high,
            'specificity_display': format_metric(specificity, spec_ci_low, spec_ci_high),
            'n_fp_clusters': n_fp_clusters,
            'mean_no_fp_clusters': mean_no_fp_clusters,
            'mean_no_fp_clusters_ci_low': fp_ci_low,
            'mean_no_fp_clusters_ci_high': fp_ci_high,
            'mean_no_fp_clusters_display': f"{mean_no_fp_clusters:.2f} (95% CI: {fp_ci_low:.2f}-{fp_ci_high:.2f})"
        })
    return pd.DataFrame(rows)

In [ ]:
performance_table = build_performance_table(eval_stats_df, [GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED], ['noharmo', 'harmo'])
performance_table = pd.concat([performance_table, build_performance_table(eval_stats_df, [GROUP_3T_FLAIR], ['harmo'])], axis=0).reset_index(drop=True)
performance_table = pd.concat([performance_table, build_performance_table(eval_stats_df, [GROUP_7T_ADAPTED_FLAIR], ['harmo'])], axis=0).reset_index(drop=True)
performance_table = pd.concat([performance_table, build_performance_table(eval_stats_df, [GROUP_3T_UNION_7T], ['harmo'])], axis=0).reset_index(drop=True)
# show only some columns of interest, and sort
performance_table = performance_table[['analysis_group',
                                       'harmo', 
                                       'n_subjects',
                                       'n_subjects_failed_recon',
                                       'failed_recon_rate_display',
                                       'n_patients',
                                       'k_patients',
                                       'sensitivity_display',
                                       'n_mrnpatients',
                                       'k_mrnpatients',
                                       'sensitivity_mrn_display',
                                       'n_controls',
                                       'k_controls',
                                       'specificity_display',
                                       'mean_no_fp_clusters_display']]
# sort order for analysis_group: 3T, 7T_default, 7T_adapted, 3T_FLAIR, 7T_adapted_FLAIR, 3T-UNION-7T_adapted
# sort order for harmo: noharmo, harmo
performance_table.sort_values(by=['analysis_group', 'harmo'], key=lambda x: x.map({GROUP_3T: 0, GROUP_7T_DEFAULT: 1, GROUP_7T_ADAPTED: 2, GROUP_3T_FLAIR: 3, GROUP_7T_ADAPTED_FLAIR: 4, GROUP_3T_UNION_7T: 5, 'noharmo': 0, 'harmo': 1}), inplace=True)
performance_table = performance_table.T
performance_table

In [ ]:
def plot_sensitivity_specificity(eval_stats_df, 
                                 analysis_groups, 
                                 harmo_conditions, 
                                 group_hue_family, 
                                 group_tick_labels=None, 
                                 figsize=(6, 8),
                                 correction_method='fdr_bh'):
    """
    Build the 3-row (sensitivity / sensitivity in MR-negative patients / specificity) bar plot
    with 95%-CI error bars, for the given analysis groups crossed with the given harmo conditions.

    harmo_conditions=['noharmo', 'harmo'] draws paired bars per analysis group (with noharmo-vs-harmo
    McNemar brackets in addition to the pairwise, harmonized-only, across-group ones) and a second
    row of per-bar harmo x-tick labels; harmo_conditions=['harmo'] draws a single bar per analysis
    group with only the pairwise, across-group McNemar brackets.

    correction_method is passed to statsmodels' multipletests, e.g. 'fdr_bh' or 'bonferroni'.
    """
    # get the performance table for the given analysis groups and harmo conditions
    # this internally filters for matched subjects across all (harmo, analysis_group) 
    # combinations and recon_successful==1 and also calculates the number 
    # of subjects that failed recon
    performance_table = build_performance_table(eval_stats_df, analysis_groups, harmo_conditions)

    # filter the eval_stats_df to only include subjects present in all 
    # (harmo, analysis_group) combinations 
    eval_stats_df_recon = filter_to_common_subjects(eval_stats_df, analysis_groups, harmo_conditions).copy()
    eval_stats_df_recon['recon_failed'] = (eval_stats_df_recon['recon_successful'] == 0).astype(int)

    # then also filter to only include subjects that were successfully reconstructed (recon_successful==1)
    # and recon_successful==1 (needed for the McNemar tests)
    eval_stats_df = filter_to_common_subjects(eval_stats_df[eval_stats_df['recon_successful'] == 1],
                                              analysis_groups,
                                              harmo_conditions)

    hue_order = [harmo_labels[harmo] for harmo in harmo_conditions]

    fig, axes = plt.subplots(nrows=5, ncols=1, figsize=figsize, sharex=True,
                             gridspec_kw={'height_ratios': [0.15, 1, 1, 1, 1]})
    metrics = [
        ('failed_recons', 'A | Failed reconstructions'),
        ('sensitivity', 'B | Sensitivity (all patients)'),
        ('sensitivity_mrn', 'C | Sensitivity (3T MRI-negative patients)'),
        ('specificity', 'D | Specificity (healthy controls)'),
        ('mean_no_fp_clusters', 'E | Mean false positives per subject (patients and controls)'),
    ]

    # raw count columns (built by build_performance_table) backing each metric's bar
    count_columns = {
        'failed_recons': ('n_subjects_failed_recon', 'n_subjects'),
        'sensitivity': ('k_patients', 'n_patients'),
        'sensitivity_mrn': ('k_mrnpatients', 'n_mrnpatients'),
        'specificity': ('k_controls', 'n_controls'),
    }

    for ax, (metric_col, title) in zip(axes, metrics):
        # whether this metric is the one restricted to the 3T MRI-negative patients;
        # both branches below read it
        mrn = (metric_col == 'sensitivity_mrn')
        if metric_col == 'failed_recons':
            # this is a special case, we only show the raw counts, no actual 
            # bar plot
            metric_df = performance_table[['analysis_group', 'harmo', 'n_subjects_failed_recon', 'n_subjects']].copy()
            metric_df['Harmonization'] = metric_df['harmo'].map(harmo_labels)
            metric_df['value'] = metric_df['n_subjects_failed_recon']

            # draw an invisible barplot purely to reuse the same x-positions/dodge-grouping as the
            # other subplots (sharex=True); bar heights equal the actual failed-recon counts so the
            # McNemar brackets below are stacked at sensible heights
            sns.barplot(data=metric_df, x='analysis_group', y='value', hue='Harmonization',
                        order=analysis_groups, hue_order=hue_order, palette=['none'] * len(hue_order),
                        errorbar=None, legend=False, ax=ax)

            # fix the y-range before annotating so the brackets are placed relative to it, not the
            # auto-scaled range (mirrors the other 3 panels)
            ylim_top = max(metric_df['n_subjects_failed_recon'].max(), 1) * 1.6
            ax.set_ylim(0, ylim_top)

            # n_subjects_failed_recon/n_subjects doesn't depend on harmo (reconstruction success is
            # independent of harmonization), so annotate once per analysis group, centered across that
            # group's (possibly dodged) bar positions, using the first harmo condition's values
            sub = metric_df[metric_df['harmo'] == harmo_conditions[0]].set_index('analysis_group').loc[analysis_groups]
            text_y = ylim_top * 0.05
            for group_idx, analysis_group in enumerate(analysis_groups):
                group_x_positions = [container[group_idx].get_x() + container[group_idx].get_width() / 2
                                    for container in ax.containers]
                x_pos = np.mean(group_x_positions)
                failed, total = sub.loc[analysis_group, ['n_subjects_failed_recon', 'n_subjects']]
                ax.text(x_pos, text_y, f'{int(failed)}/{int(total)}',
                        ha='center', va='bottom', fontsize=10, color='black')

            for container in ax.containers:
                for patch in container:
                    patch.set_visible(False)

            # McNemar tests for recon failure; use eval_stats_df_recon, which (unlike eval_stats_df
            # used by the other 3 panels) retains subjects whose reconstruction failed
            # display bracket only if p<0.05
            outcome_col = 'recon_failed'
            pairs, pvalues = [], []

            for group_a, group_b in itertools.combinations(analysis_groups, 2):
                df_a = eval_stats_df_recon[(eval_stats_df_recon['harmo'] == 'harmo') & (eval_stats_df_recon['analysis_group'] == group_a)]
                df_b = eval_stats_df_recon[(eval_stats_df_recon['harmo'] == 'harmo') & (eval_stats_df_recon['analysis_group'] == group_b)]
                _, _, _, pvalue = mcnemar_paired(df_a, df_b, outcome_col)
                if pvalue < 0.05:
                    pairs.append(((group_a, harmo_labels['harmo']), (group_b, harmo_labels['harmo'])))
                    pvalues.append(pvalue)

            if len(pairs) > 0:
                annotator = Annotator(ax, pairs, plot='barplot', data=metric_df, x='analysis_group', y='value',
                                        hue='Harmonization', order=analysis_groups, hue_order=hue_order, verbose=0)
                annotator.configure(test=None,
                                    text_format='star',
                                    loc='inside',
                                    line_offset_to_group=0.01,
                                    line_offset=0.005,
                                    line_height=0.02,
                                    fontsize=9,
                                    color='gray')
                annotator.pvalue_format.pvalue_thresholds = [
                    (0.05, '*'),
                    (1, 'ns')
                ]
                annotator.set_pvalues(pvalues)
                annotator.annotate()

            ax.set_yticks([])
            ax.set_ylabel('')
            ax.set_xlabel('')
            ax.set_title(title, fontsize=10, ha='left', x=0.0, y=1.02, fontweight='bold')
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.tick_params(left=False)
            ax.tick_params(axis='x', which='both', length=0)

            continue
        elif metric_col == 'mean_no_fp_clusters':
            # not a proportion, so this gets its own bar plot instead of reusing the
            # percentage scaling used for sensitivity/specificity below
            metric_df = performance_table[['analysis_group', 'harmo', 'mean_no_fp_clusters',
                                           'mean_no_fp_clusters_ci_low', 'mean_no_fp_clusters_ci_high',
                                           'n_fp_clusters']].copy()
            metric_df['Harmonization'] = metric_df['harmo'].map(harmo_labels)
            metric_df['value'] = metric_df['mean_no_fp_clusters']
            metric_df['ci_low'] = metric_df['mean_no_fp_clusters_ci_low']
            metric_df['ci_high'] = metric_df['mean_no_fp_clusters_ci_high']

            sns.barplot(data=metric_df, x='analysis_group', y='value', hue='Harmonization',
                        order=analysis_groups, hue_order=hue_order, palette=['lightgray'] * len(hue_order),
                        errorbar=None, legend=False, ax=ax)

            # fix the y-range before annotating so the "n=" labels are placed relative to it, not
            # the auto-scaled range (mirrors the other panels); guard against an all-NaN metric_df
            # (e.g. for UNION conditions, where fp_clusters is not meaningful)
            ylim_top = np.nanmax([metric_df['ci_high'].max(skipna=True), 1]) * 1.3

            for container, harmo in zip(ax.containers, harmo_conditions):
                sub = metric_df[metric_df['harmo'] == harmo].set_index('analysis_group').loc[analysis_groups]
                for patch, analysis_group in zip(container, analysis_groups):
                    color = _with_saturation(group_hue_family[analysis_group], harmo_saturation[harmo])
                    patch.set_facecolor(color)
                    x_pos = patch.get_x() + patch.get_width() / 2
                    value, ci_low, ci_high, n = sub.loc[analysis_group, ['value', 'ci_low', 'ci_high', 'n_fp_clusters']]
                    if pd.isna(value):
                        continue
                    height = patch.get_height()
                    yerr = [[value - ci_low], [ci_high - value]]
                    ax.errorbar([x_pos], [height], yerr=yerr, fmt='none', ecolor=_with_value(color, 0.6), capsize=3, linewidth=1)
                    ax.text(x_pos, 0, f'{sub.loc[analysis_group, "value"]:.1f}', ha='center', va='bottom', fontsize=8, color='black')

            ax.set_ylim(0, ylim_top)
            ax.set_title(title, fontsize=10, ha='left', x=0.0, y=1.02, fontweight='bold')
            ax.set_ylabel('Mean count')
            ax.set_xlabel('')
            ax.tick_params(axis='x', which='both', length=0)
            ax.grid(axis='y', color='black', alpha=0.15, linewidth=0.8, zorder=0)
            ax.set_axisbelow(True)
            for spine in ['top', 'right']:
                ax.spines[spine].set_visible(False)

            # compute pairwise Wilcoxon signed rank tests for the mean_no_fp_clusters metric, within the harmonized group, between all analysis groups
            pairs, pvalues = [], []
            if len(harmo_conditions) > 1:
                # McNemar test for noharmo vs harmo, paired by site_subj_id, within each analysis group
                # mcnemar_paired() merged on site_subj_id and verifies a 1:1 match
                for analysis_group in analysis_groups:
                    df_noharmo = eval_stats_df[(eval_stats_df['harmo'] == 'noharmo') & (eval_stats_df['analysis_group'] == analysis_group)]
                    df_harmo = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == analysis_group)]
                    if mrn:
                        df_noharmo = df_noharmo[df_noharmo['category_3T_MR_negative'] == 1]
                        df_harmo = df_harmo[df_harmo['category_3T_MR_negative'] == 1]
                    _, pvalue = wilcoxon(df_noharmo['fp_clusters'], df_harmo['fp_clusters'])
                    pairs.append(((analysis_group, harmo_labels['noharmo']), (analysis_group, harmo_labels['harmo'])))
                    pvalues.append(pvalue)

            # pairwise Wilcoxon tests between the analysis groups, within the harmonized group; their
            # wider x-range makes statannotations auto-stack them above the noharmo-vs-harmo brackets
            for group_a, group_b in itertools.combinations(analysis_groups, 2):
                df_a = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == group_a)]
                df_b = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == group_b)]
                if mrn:
                    df_a = df_a[df_a['category_3T_MR_negative'] == 1]
                    df_b = df_b[df_b['category_3T_MR_negative'] == 1]
                _, pvalue = wilcoxon(df_a['fp_clusters'], df_b['fp_clusters'])
                pairs.append(((group_a, harmo_labels['harmo']), (group_b, harmo_labels['harmo'])))
                pvalues.append(pvalue)

        else:
            k_col, n_col = count_columns[metric_col]
            metric_df = performance_table[['analysis_group', 'harmo', metric_col, f'{metric_col}_ci_low', f'{metric_col}_ci_high', k_col, n_col]].copy()
            metric_df['Harmonization'] = metric_df['harmo'].map(harmo_labels)
            metric_df['value'] = metric_df[metric_col] * 100
            metric_df['ci_low'] = metric_df[f'{metric_col}_ci_low'] * 100
            metric_df['ci_high'] = metric_df[f'{metric_col}_ci_high'] * 100

            sns.barplot(data=metric_df, x='analysis_group', y='value', hue='Harmonization',
                        order=analysis_groups, hue_order=hue_order, palette=['lightgray'] * len(hue_order),
                        errorbar=None, legend=False, ax=ax)

            # recolor each bar by (analysis_group, harmo) and overlay asymmetric 95%-CI error bars in the same color
            for container, harmo in zip(ax.containers, harmo_conditions):
                sub = metric_df[metric_df['harmo'] == harmo].set_index('analysis_group').loc[analysis_groups]
                for patch, analysis_group in zip(container, analysis_groups):
                    color = _with_saturation(group_hue_family[analysis_group], harmo_saturation[harmo])
                    patch.set_facecolor(color)
                    x_pos = patch.get_x() + patch.get_width() / 2
                    height = patch.get_height()
                    value, ci_low, ci_high, k, n = sub.loc[analysis_group, ['value', 'ci_low', 'ci_high', k_col, n_col]]
                    yerr = [[value - ci_low], [ci_high - value]]
                    ax.errorbar([x_pos], [height], yerr=yerr, fmt='none', ecolor=_with_value(color, 0.6), capsize=3, linewidth=1)
                    # raw counts behind the percentage (e.g. true positives / patients), near the bar's base
                    ax.text(x_pos, 2, f'{int(k)}/{int(n)}', ha='center', va='bottom', fontsize=8, color='black')

            # fix the y-range before annotating so the brackets are placed relative to it, not the auto-scaled range
            ax.set_ylim(0, 90)

            outcome_col = 'tn_control' if metric_col == 'specificity' else 'tp_patient'
            pairs, pvalues = [], []

            if len(harmo_conditions) > 1:
                # McNemar test for noharmo vs harmo, paired by site_subj_id, within each analysis group
                # mcnemar_paired() merged on site_subj_id and verifies a 1:1 match
                for analysis_group in analysis_groups:
                    df_noharmo = eval_stats_df[(eval_stats_df['harmo'] == 'noharmo') & (eval_stats_df['analysis_group'] == analysis_group)]
                    df_harmo = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == analysis_group)]
                    if mrn:
                        df_noharmo = df_noharmo[df_noharmo['category_3T_MR_negative'] == 1]
                        df_harmo = df_harmo[df_harmo['category_3T_MR_negative'] == 1]
                    _, _, _, pvalue = mcnemar_paired(df_noharmo, df_harmo, outcome_col)
                    pairs.append(((analysis_group, harmo_labels['noharmo']), (analysis_group, harmo_labels['harmo'])))
                    pvalues.append(pvalue)

            # pairwise McNemar tests between the analysis groups, within the harmonized group; their
            # wider x-range makes statannotations auto-stack them above the noharmo-vs-harmo brackets
            for group_a, group_b in itertools.combinations(analysis_groups, 2):
                df_a = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == group_a)]
                df_b = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == group_b)]
                if mrn:
                    df_a = df_a[df_a['category_3T_MR_negative'] == 1]
                    df_b = df_b[df_b['category_3T_MR_negative'] == 1]
                _, _, _, pvalue = mcnemar_paired(df_a, df_b, outcome_col)
                pairs.append(((group_a, harmo_labels['harmo']), (group_b, harmo_labels['harmo'])))
                pvalues.append(pvalue)

        # correct p-values for multiple testing
        _, pvalues, _, _ = multipletests(pvalues, alpha=0.05, method=correction_method)

        pairs_filtered, pvalues_filtered = [], []
        for pair, pvalue in zip(pairs, pvalues):
            if pvalue < 0.05:
                pairs_filtered.append(pair)
                pvalues_filtered.append(pvalue)
        pairs, pvalues = pairs_filtered, pvalues_filtered

        if len(pairs) > 0:
            # print all pairs
            for pair, pvalue in zip(pairs, pvalues):
                print(f"{pair[0]} vs {pair[1]}: p={pvalue:.4f}")

            annotator = Annotator(ax, pairs, plot='barplot', data=metric_df, x='analysis_group', y='value',
                                    hue='Harmonization', order=analysis_groups, hue_order=hue_order, verbose=0)
            annotator.configure(test=None,
                                text_format='star',
                                show_test_name=False,
                                loc='inside',
                                line_offset_to_group=0.01,
                                line_offset=0.005,
                                line_height=0.02,
                                fontsize=9,
                                color='gray')
            annotator.pvalue_format.pvalue_thresholds = [
                (0.05, '*'),
                (1, 'ns')
            ]
            annotator.set_pvalues(pvalues)
            annotator.annotate()


        if metric_col in ['sensitivity', 'sensitivity_mrn', 'specificity']:
            ax.set_ylabel('%')
            # cap the visible ticks/gridlines/y-spine at 90 even though ylim may extend higher
            ax.set_yticks(np.arange(0, 91, 20))
            ax.spines['left'].set_bounds(0, 90)
            ax.set_title(title, fontsize=10, ha='left', x=0.0, y=1.02, fontweight='bold')
            ax.set_xlabel('')
            ax.tick_params(axis='x', which='both', length=0)
            # set_ylabel() centers on the full (possibly annotation-expanded) ylim by default; re-anchor
            # it to the middle of the visible 0-90 band instead. This takes over from matplotlib's
            # automatic label placement, so the x-offset is now a fixed axes-fraction value rather than
            # one that adapts to tick-label width
            ax.yaxis.set_label_coords(-0.06, 45, transform=mtransforms.blended_transform_factory(ax.transAxes, ax.transData))
            ax.grid(axis='y', color='black', alpha=0.15, linewidth=0.8, zorder=0)
            ax.set_axisbelow(True)
            for spine in ['top', 'right']:
                ax.spines[spine].set_visible(False)
        elif metric_col == 'mean_no_fp_clusters':
            pass



    axes[-1].set_xticks(range(len(analysis_groups)))
    axes[-1].set_xticklabels([(group_tick_labels or {}).get(analysis_group, analysis_group) for analysis_group in analysis_groups])

    if len(harmo_conditions) > 1:
        # add a second row of x-axis labels naming the harmo condition of each individual bar
        minor_positions, minor_labels = [], []
        for container, harmo in zip(axes[-1].containers, harmo_conditions):
            for patch in container:
                minor_positions.append(patch.get_x() + patch.get_width() / 2)
                minor_labels.append(harmo_labels[harmo])

        axes[-1].set_xticks(minor_positions, minor=True)
        axes[-1].set_xticklabels(minor_labels, minor=True, fontsize=8, rotation=45, ha='right', rotation_mode='anchor')
        axes[-1].tick_params(axis='x', which='minor', pad=2, length=0)
        axes[-1].tick_params(axis='x', which='major', pad=40, length=0, labelsize=10)

    plt.tight_layout(h_pad=1.5)
    return fig, axes, performance_table

### Figure 2

In [ ]:
fig, axes, performance_table = plot_sensitivity_specificity(
    eval_stats_df,
    analysis_groups=[GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED],
    harmo_conditions=['noharmo', 'harmo'],
    group_hue_family=main_group_hue_family,
)

### list cases as example plot candidates

In [ ]:
# list possible subjects to plot
eval_stats_df_3T7T = eval_stats_df[(eval_stats_df['analysis_group'].isin([GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED])) & (eval_stats_df['harmo'] == 'harmo')]
eval_stats_df_3T7T = filter_to_common_subjects(eval_stats_df_3T7T, [GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED], ['harmo'])
eval_stats_df_3T7T_patients = eval_stats_df_3T7T.pivot(index=['site_subj_id', 'confirmed_histology'], columns='analysis_group', values='tp_patient').dropna().reset_index()
eval_stats_df_3T7T_controls = eval_stats_df_3T7T.pivot(index=['site_subj_id', 'confirmed_histology'], columns='analysis_group', values='tn_control').dropna().reset_index()
print(eval_stats_df_3T7T_patients.head())
print(eval_stats_df_3T7T_controls.head())

In [ ]:
# patients positive at 3T and 7T adapted
eval_stats_df_3T7T_patients[(eval_stats_df_3T7T_patients['3T'] == 1) & (eval_stats_df_3T7T_patients['7T adapted'] == 1)]

In [ ]:
# patients negative at 3T and positive at either 7T adapted or 7T default
eval_stats_df_3T7T_patients[(eval_stats_df_3T7T_patients[GROUP_3T] == 0) & ((eval_stats_df_3T7T_patients[GROUP_7T_DEFAULT] == 1) | (eval_stats_df_3T7T_patients[GROUP_7T_ADAPTED] == 1))]

In [ ]:
# patients positive at 3T and negative at either 7T adapted or 7T default
eval_stats_df_3T7T_patients[(eval_stats_df_3T7T_patients[GROUP_3T] == 1) & ((eval_stats_df_3T7T_patients[GROUP_7T_DEFAULT] == 0) | (eval_stats_df_3T7T_patients[GROUP_7T_ADAPTED] == 0))]

In [ ]:
# patients negative at all conditions but histopathologically verified
eval_stats_df_3T7T_patients[~eval_stats_df_3T7T_patients[[GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED]].any(axis=1) & eval_stats_df_3T7T_patients['confirmed_histology'] == 1]

In [ ]:
# controls negative at 3T but with false positive at either 7T adapted or 7T default
eval_stats_df_3T7T_controls[(eval_stats_df_3T7T_controls[GROUP_3T] == 1) & ((eval_stats_df_3T7T_controls[GROUP_7T_DEFAULT] == 0) | (eval_stats_df_3T7T_controls[GROUP_7T_ADAPTED] == 0))]

In [ ]:
# controls positive in all conditions 
eval_stats_df_3T7T_controls[(eval_stats_df_3T7T_controls[GROUP_3T] == 0) & (eval_stats_df_3T7T_controls[GROUP_7T_DEFAULT] == 0) & (eval_stats_df_3T7T_controls[GROUP_7T_ADAPTED] == 1)]

### union/combined 3T + 7T

### Supplementary Figure 2

In [ ]:
fig_union, axes_union, performance_table_union = plot_sensitivity_specificity(
    eval_stats_df,
    analysis_groups=[GROUP_3T, GROUP_7T_ADAPTED, GROUP_3T_UNION_7T],
    harmo_conditions=['harmo'],
    group_hue_family={GROUP_3T: "#a9c8f0", GROUP_7T_ADAPTED: '#f4b78a', GROUP_3T_UNION_7T: '#a8dab5'},
    group_tick_labels={GROUP_3T_UNION_7T: '3T + 7T adapted'},
)

### Supplementary Figure 3

In [ ]:
fig_3tflair, axes_3tflair, performance_table_3tflair = plot_sensitivity_specificity(
    eval_stats_df,
    analysis_groups=[GROUP_3T, GROUP_3T_FLAIR],
    harmo_conditions=['harmo'],
    group_hue_family={GROUP_3T: "#a9c8f0", GROUP_3T_FLAIR: _with_value('#a9c8f0', 0.8)},
    group_tick_labels={GROUP_3T: '3T T1w only', GROUP_3T_FLAIR: '3T T1w + FLAIR'},
)

In [ ]:
fig_7tflair, axes_7tflair, performance_table_7tflair = plot_sensitivity_specificity(
    eval_stats_df,
    analysis_groups=[GROUP_3T, GROUP_7T_ADAPTED, GROUP_7T_ADAPTED_FLAIR],
    harmo_conditions=['harmo'],
    group_hue_family={GROUP_3T: "#a9c8f0", GROUP_7T_ADAPTED: '#f4b78a', GROUP_7T_ADAPTED_FLAIR: _with_value('#f4b78a', 0.8)},
    group_tick_labels={GROUP_3T: '3T T1w only', GROUP_7T_ADAPTED: '7T T1w only', GROUP_7T_ADAPTED_FLAIR: '7T T1w + FLAIR'},
)

#plt.show()